<a href="https://colab.research.google.com/github/Yusufkotavom/AI-Content-Image-Generator-SaaS/blob/master/Yt_upload_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q streamlit pyngrok google-api-python-client google-auth-oauthlib google-auth-httplib2 mutagen pillow groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 49.7 MB/s eta 0:00:00




```
# Ini diformat sebagai kode
```

# DOWNLOAD

In [ ]:
!pip install -q yt-dlp mutagen beautifulsoup4 requests
!apt-get install -q -y ffmpeg

import os, time, random, subprocess, requests, shutil, json
from bs4 import BeautifulSoup
from mutagen.mp3 import MP3

# ==========================================
# KONFIGURASI SMART FACTORY V5.2 (ULTIMATE)
# ==========================================
NEGARA = 'global'            # Ganti ke 'global' jika ingin scrape chart global
TOTAL_SCRAPE = 100
TOTAL_LAGU_PER_VIDEO = 25
JUMLAH_VERSI = 12
DIR_DL = "downloads"
DIR_FOOTAGE = "/content/drive/MyDrive/yt_mp3/footage"

# --- Folder Produksi & Master Cache ---
WAKTU_PRODUKSI = time.strftime("%Y%m%d_%H%M")
NAMA_FOLDER = f"PRODUKSI_{NEGARA.upper()}_{WAKTU_PRODUKSI}"
DIR_DRIVE_PROD = f"/content/drive/MyDrive/yt_mp3/hasil_produksi/{NAMA_FOLDER}"
DIR_DRIVE_MASTER = f"/content/drive/MyDrive/yt_mp3/master_lagu_{NEGARA.lower()}"

def log(step, pesan):
    waktu = time.strftime("%H:%M:%S")
    print(f"[{waktu}] [{step}] {pesan}")

def format_waktu(detik):
    m, s = divmod(int(detik), 60)
    h, m = divmod(m, 60)
    if h > 0: return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

# 0. STERILISASI & PERSIAPAN FOLDER
log("SETUP", f"Menyiapkan Batch: {NAMA_FOLDER}...")
if os.path.exists(DIR_DL): shutil.rmtree(DIR_DL)
for d in [DIR_DL, DIR_DRIVE_PROD, DIR_FOOTAGE, DIR_DRIVE_MASTER]:
    os.makedirs(d, exist_ok=True)

# 1. SCRAPING DATA KWORB
log("SCRAPE", f"Mencari kandidat lagu dari Top {TOTAL_SCRAPE} Spotify {NEGARA.upper()}...")
try:
    res = requests.get(f"https://kworb.net/spotify/country/{NEGARA}_daily.html", headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
    soup = BeautifulSoup(res.text, 'html.parser')
    baris_tabel = soup.find('tbody').find_all('tr')

    daftar_target = []
    for i, baris in enumerate(baris_tabel[:TOTAL_SCRAPE]):
        kolom = baris.find_all('td')
        if len(kolom) >= 3:
            teks = kolom[2].text.strip()
            artis, judul = teks.split(' - ', 1) if ' - ' in teks else ("Unknown", teks)
            daftar_target.append({'Rank': i + 1, 'Artis': artis, 'Judul': judul, 'Target': f"{artis} - {judul} (Official Audio)"})
except Exception as e:
    log("ERROR", f"Gagal scraping: {e}"); exit()

# 2. SMART CACHE DOWNLOADER (ANTI-DUPLICATE & AUTO-SAVE)
log("DOWNLOAD", f"Sinkronisasi {len(daftar_target)} lagu ke Master Database...")
for item in daftar_target:
    artis_aman = item['Artis'].replace('/', '_').replace(':', '').replace('"', '')
    judul_aman = item['Judul'].replace('/', '_').replace(':', '').replace('"', '')

    # Nama master tanpa nomor urut (agar bisa dicari lintas waktu)
    nama_master = f"{artis_aman} - {judul_aman}.mp3"
    path_master = os.path.join(DIR_DRIVE_MASTER, nama_master)

    # Nama lokal pakai nomor urut untuk kemudahan perakitan FFmpeg
    nama_lokal = f"{item['Rank']:02d} - {artis_aman} - {judul_aman}.mp3"
    path_lokal = os.path.join(DIR_DL, nama_lokal)

    if os.path.exists(path_master):
        shutil.copy(path_master, path_lokal)
        log("CACHE", f"⏭️ Skip (Ambil dari Drive): {nama_master}")
    else:
        log("DOWNLOAD", f"⬇️ Baru: {nama_master}")
        subprocess.run(['yt-dlp', '-x', '--audio-format', 'mp3', '-o', path_lokal, f"ytsearch1:{item['Target']}"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        # AUTO-SAVE: Begitu selesai didownload, simpan copynya ke Master Drive
        if os.path.exists(path_lokal):
            shutil.copy(path_lokal, path_master)

# 3. PROSES PABRIKAN VERSI MASSAL
log("FACTORY", "Download selesai! Memulai perakitan super cepat...")
files_mp3 = [f for f in os.listdir(DIR_DL) if f.endswith('.mp3')]
top_5_master = [f for f in files_mp3 if f.split(' - ')[0].isdigit() and 1 <= int(f.split(' - ')[0]) <= 5]
rest_master = [f for f in files_mp3 if f not in top_5_master]

footages = [f for f in os.listdir(DIR_FOOTAGE) if f.lower().endswith(('.mp4', '.mov', '.mkv'))]
if not footages: log("WARNING", "Video footage kosong! Hasil hanya berupa file MP3 panjang.")

for v in range(1, JUMLAH_VERSI + 1):
    log("BATCH", f"========== MERAKIT VOL {v}/{JUMLAH_VERSI} ==========")

    video_input = os.path.join(DIR_FOOTAGE, random.choice(footages)) if footages else None
    jumlah_butuh = TOTAL_LAGU_PER_VIDEO - 5
    rest_v = random.sample(rest_master, min(jumlah_butuh, len(rest_master)))
    top_5_v = top_5_master.copy()

    random.shuffle(top_5_v); random.shuffle(rest_v)
    playlist = top_5_v + rest_v

    path_audio_tmp = os.path.join(DIR_DL, f"temp_v{v}.mp3")
    path_txt = os.path.join(DIR_DL, f"list_v{v}.txt")
    teks_timeline = "00:00 - Start\n"
    total_detik = 0

    with open(path_txt, 'w', encoding='utf-8') as f:
        for lagu in playlist:
            safe_name = lagu.replace("'", "'\\''")
            f.write(f"file '{safe_name}'\n")
            judul_bersih = lagu.split(' - ', 1)[-1].replace('.mp3','')
            teks_timeline += f"{format_waktu(total_detik)} - {judul_bersih}\n"
            try: total_detik += MP3(os.path.join(DIR_DL, lagu)).info.length
            except: pass

    log("FFMPEG", f"Menjahit audio (Speed Mode)...")
    subprocess.run(['ffmpeg', '-f', 'concat', '-safe', '0', '-i', path_txt, '-c', 'copy', path_audio_tmp, '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    base_name = f"VIDEO_{NEGARA.upper()}_VOL_{v}"
    final_video_name = f"{base_name}.mp4"
    path_video_lokal = os.path.join(DIR_DL, final_video_name)

    if video_input:
        log("FFMPEG", "Menyatukan MP3 & Visual (Video Copy + YouTube Time Patch)...")
        subprocess.run([
            'ffmpeg', '-stream_loop', '-1', '-i', video_input, '-i', path_audio_tmp,
            '-c:v', 'copy',
            '-c:a', 'aac', '-b:a', '192k',
            '-shortest',
            '-fflags', '+genpts',
            path_video_lokal, '-y'
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # ---------------------------------------------------------
    # PEMINDAHAN & PEMBUATAN METADATA JSON
    # ---------------------------------------------------------
    file_final = path_video_lokal if video_input else path_audio_tmp
    if os.path.exists(file_final):
        target_drive = os.path.join(DIR_DRIVE_PROD, final_video_name) if video_input else os.path.join(DIR_DRIVE_PROD, f"{base_name}.mp3")
        shutil.move(file_final, target_drive)

        meta_json = {
            "title": f"{base_name}",
            "description": teks_timeline,
            "tags": ["musik", "pop", "spotify", "2026", NEGARA]
        }
        with open(os.path.join(DIR_DRIVE_PROD, f"{base_name}.json"), 'w', encoding='utf-8') as fj:
            json.dump(meta_json, fj, indent=4)

        log("DRIVE", f"✅ {base_name} Sukses diproduksi!")

    if os.path.exists(path_txt): os.remove(path_txt)
    if os.path.exists(path_audio_tmp): os.remove(path_audio_tmp)

log("DONE", f"🎉 PABRIK SELESAI! Hasil siap di-scan di Nexus Commander.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 45.5 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/

KeyboardInterrupt: 

# Bagian Baru

In [ ]:
# ==========================================
# 0. AUTO-INSTALLER & DEPENDENCIES
# ==========================================
!pip install -q yt-dlp mutagen beautifulsoup4 requests
!apt-get install -q -y ffmpeg

import os, time, random, subprocess, requests, shutil, json
from mutagen.mp3 import MP3

# ==========================================
# KONFIGURASI SMART FACTORY V5.3
# ==========================================
TOTAL_LAGU_PER_VIDEO = 25
JUMLAH_VERSI = 12         # Jumlah video yang akan dibuat sekaligus
DIR_DL = "downloads"
DIR_FOOTAGE = "/content/drive/MyDrive/yt_mp3/footage"

# --- Folder Produksi & Master Cache ---
WAKTU_PRODUKSI = time.strftime("%Y%m%d_%H%M")
NAMA_FOLDER = f"PRODUKSI_NOSTALGIA_{WAKTU_PRODUKSI}"
DIR_DRIVE_PROD = f"/content/drive/MyDrive/yt_mp3/hasil_produksi/{NAMA_FOLDER}"
DIR_DRIVE_MASTER = f"/content/drive/MyDrive/yt_mp3/master_lagu_nostalgia"

def log(step, pesan):
    waktu = time.strftime("%H:%M:%S")
    print(f"[{waktu}] [{step}] {pesan}")

def format_waktu(detik):
    m, s = divmod(int(detik), 60)
    h, m = divmod(m, 60)
    if h > 0: return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

# 0. SETUP FOLDER
log("SETUP", f"Menyiapkan Batch Produksi: {NAMA_FOLDER}...")
if os.path.exists(DIR_DL): shutil.rmtree(DIR_DL)
for d in [DIR_DL, DIR_DRIVE_PROD, DIR_FOOTAGE, DIR_DRIVE_MASTER]:
    os.makedirs(d, exist_ok=True)

# ==========================================
# 1. INPUT DAFTAR LAGU CUSTOM
# ==========================================
DATA_LAGU_RAW = """
11 Januari – Gigi
Aku Ada Karena Kau Ada – Radja
Akhirnya Ku Menemukanmu – NaFF
Andai – Gigi
Arti Cinta – Ari Lasso
Bagaikan Langit – POTRET
Berdua Lebih Baik – Acha Septriasa
Betapa – Sheila On 7
Bila Aku Jatuh Cinta – Nidji
Bila Rasaku Ini Rasamu – Kerispatih
Bukan Permainan – Gita Gutawa
Bukannya Aku Takut – Juliette
Cinta Pertama (Sunny) – Bunga Citra Lestari
Cinta Tak Bersyarat – Element
Cinta Terakhir – Ari Lasso
Cinta Terlarang – The Virgin
Cobalah – Hijau Daun
Demi Waktu – Ungu
Dua Cincin – Hello
Empat Mata – D'Bagindas
Gantung – Melly Goeslaw
Hanya Ingin Kau Tahu – Repvblik
Hapus Aku – Nidji
Hidupku Kan Damaikan Hatimu – Caffeine
Ijinkan Aku Menyayangimu – Iwan Fals
Jadikan Aku Yang Kedua – Astrid
Jantung Hati – Lyla
Jika (feat. Ari Lasso) – Melly Goeslaw feat. Ari Lasso
Karma – Cokelat
Kasih – Salju
Kau Masih Kekasihku – NaFF
Kehadiranmu – Vagetoz
Kembali Pulang – Kangen Band
Kenangan Terindah – SAMSONS
Kisah Sedih Di Hari Minggu – Marshanda
Kucinta Kau Apa Adanya (Aku Mau) – Once Mekel
Laguku – Ungu
Lelaki Cadangan – T2
Luka Disini – Ungu
Main Hati – Andra & The Backbone
Manusia Bodoh – Ada Band
Mantan Kekasih – Lyla
Melepasmu – Drive
Menanti Sebuah Jawaban – Padi
Menghapus Jejakmu – Noah
Munajat Cinta – The Rock
Naluri Lelaki – SAMSONS
OK – T2
Pelan Pelan Saja – Kotak
Pelangi Di Matamu – Jamrud
Permintaan Hati – Letto
Ayat-Ayat Cinta – Rossa
Ular Berbisa – Hello
Pujaan Hati – Kangen Band
Ruang Rindu – Letto
Saat Indah Bersamamu – Ungu
Saat Kau Pergi – Vagetoz
Saat Kau Pergi – Bunga Citra Lestari
Saat Terakhir – ST12
Sampai Mati – Putih
Sebatas Mimpi – NANO
Sebelum Cahaya – Letto
Selamat Tinggal – Five Minutes
Selamanya Cinta – D'Cinnamons
Semakin Ku Kejar Semakin Kau Jauh (SKSJ) – Five Minutes
Semua Tentang Kita – Noah
Sephia – Sheila On 7
Setia – Jikustik
Sobat – Padi
Suara (Ku Berharap) – Hijau Daun
Suara Hati – Ungu
Sudah – Nidji
Tak Bisakah – Noah
Tak Lekang Oleh Waktu – Kerispatih
Tapi Bukan Aku – Kerispatih
Tentang Aku, Kau dan Dia – Kangen Band
Ingatlah Hari Ini – Project Pop
Terbang Bersamaku – Kangen Band
Terendap Laraku – NaFF
Yang Terdalam – Noah
Yank – Wali
Aku Pasti Kembali – Pasto
Bila Kau Tak Disampingku – Sheila On 7
Cinta Tak Harus Memiliki – ST12
Dan... – Sheila On 7
Demi Cinta – Kerispatih
Haruskah Ku Mati – Ada Band
Jaga Slalu Hatimu – Seventeen
Kangen – Dewa 19
Kasih Tak Sampai – Padi
Kau Auraku – Ada Band
Lagu Rindu – Kerispatih
Mungkinkah – Stinky
Pandangi Langit Malam Ini – Jikustik
Penjaga Hati – Ari Lasso
Sempurna – Andra & The Backbone
Terabaikan – Garis Temu
(BAM) Betapa Aku Mencintaimu – Vagetoz
Ada Apa Denganmu – Noah
Aisah – Five Minutes
Aishiteru – Zivilia
Aku Cinta Kau Dan Dia – The Rock
Aku Masih Sayang – ST12
Angin – Radja
Antara Ada dan Tiada – Utopia
Anugerah Terindah Yang Pernah Kumiliki – Sheila On 7
Apa Salahku – D’MASIV
Apa Yang Terjadi – D'Bagindas
Arjuna – Dewa
Arti Sahabat – Nidji
Benci Untuk Mencinta – Naif
Beraksi – Kotak
Bersama Bintang – Drive
Bersamamu – Vierra
Bertahan – Five Minutes
Biarlah – Killing Me Inside
Bila Nanti Kau Milikku – NaFF
Bintang – animA
Cemburu – Dewa 19
Bintang Di Surga – Noah
Boyband – Tipe-X
Bukan Diriku – SAMSONS
C.I.N.T.A – D'Bagindas
Cari Pacar Lagi – ST12
Ceria – J-Rocks
Child – Nidji
Cinderella – Radja
Cinta ...Bersabarlah – Letto
Cinta Dan Benci – Geisha
Cinta Gila – Ungu
Cinta Ini Membunuhku – D’MASIV
Cinta Monyet – Goliath
Cinta Sampai Disini – D’MASIV
Cinta Tak Pernah Sama – Nidji
Cinta 'Kan Membawamu Kembali – Dewa 19
Cobalah Kau Mengerti – J-Rocks
Cobalah Mengerti – Noah
Dengarkan Curhatku – Vierra
Dewi – Alexa
Di Atas Awan – Nidji
Di Atas Normal – Noah
Di Balik Awan – Noah
Diam Tanpa Kata – D’MASIV
Diantara Kalian – D’MASIV
Dirimu Satu – Ungu
Disaat Aku Mencintaimu – Dadali
Disini Untukmu – Ungu
Doy – Kangen Band
Engkau – Nidji
Fallin' In Love – J-Rocks
Gak Ada Waktu Ke Laut Aja Lo – Radja
Genit – Tipe-X
Gila - Gilaan – The Changcuters
Hadapi Dengan Senyuman – Dewa
Hal Terindah – Seventeen
Hampa Hatiku – Ungu
Hanya Kamu Yang Bisa – Tiket
Hari Bersamanya – Sheila On 7
Mengenangmu – Kerispatih
Hari Yang Indah – NTRL
Harmony – Padi
Harusnya Aku – Armada
Heaven – Nidji
Hingga Akhir Waktu – NINEBALL
Hitamku – Andra & The Backbone
I Love U Bibeh – The Changcuters
Itu Aku – Sheila On 7
Jadi Yang Kuinginkan – Vierra
Jalanmu Bukan Jalanku – Andra & The Backbone
Jambu (Janjimu Busuk) – Matta
Jangan Kau Lepas – Alexa
Jangan Lupakan – Nidji
Jangan Menyerah – D’MASIV
Jangan Pernah Berubah – ST12
Jangan Pernah Selingkuh (JPS) – Angkasa
Jauh Mimpiku – Noah
Jatuh Cinta – Marvells
Jika Cinta Dia – Geisha
Jika Kami Bersama – Superman Is Dead
Jujur – Radja
Kamu – Killing Me Inside
Kamu Ngga Sendirian – Tipe-X
Kamu Yang Pertama – Geisha
Karena Kamu – Geisha
Karena Wanita (Ingin Dimengerti) – Ada Band
Kau dan Aku – Nidji
Kekasih Gelapku – Ungu
Kesepian – Vierra
Keterlaluan – The Potters
Ketahuan – Matta
Kisah Cintaku – Noah
Kota Mati – Noah
Ku Katakan Dengan Indah – Noah
Ku Tak Bisa – Slank
Ku Takkan Bisa – Nidji
Kupu - Kupu Malam – Noah
Lagi...Dan Lagi... – Andra & The Backbone
Langit Tak Mendengar – Noah
Lapang Dada – Sheila On 7
Laskar Cinta (Chapter One) – Dewa 19
Laskar Pelangi – Nidji
Lebih Dari Bintang – Lyla
Lubang Di Hati – Letto
Mabuk Cinta – Armada
Madu Tiga – T.R.I.A.D
Magic – Lyla
Manusia Biasa – Radja
Manusia Sempurna – Nidji
Masih Cinta – Kotak
Masih Disini Masih Denganmu (MD2) – Goliath
Membuatmu Cinta Padaku – Asbak Band
Melati Aku Benci Kamu – Tipe-X
Melompat Lebih Tinggi – Sheila On 7
Menemukanmu – Seventeen
Menunggumu – Noah
Meraih Mimpi – J-Rocks
Merindukanmu – D’MASIV
Mimpi Yang Sempurna – Noah
Mobil Balap – Naif
Mungkin Nanti – Noah
Musnah – Andra & The Backbone
My Facebook – Gigi
Nakal – Gigi
Natural – D’MASIV
Nyaman – D’MASIV
P.U.S.P.A. – ST12
Papa Rock N Roll – The Dance Company
Patah Hati – Radja
Pejantan Tangguh – Sheila On 7
Pemain Cinta – Ada Band
Pemilik Hati – Armada
Pemuja Rahasia – Sheila On 7
Pemujamu – Ada Band
Percaya Padaku – Ungu
Pergilah Kasih – D’MASIV
Perih – Vierra
Pertempuran Hati – NTRL
Pesona – Numata
Pria Idaman Wanita – The Changcuters
Putri Iklan – ST12
Racun Dunia – The Changcuters
Rahasia Hati – Element
Raja Jatuh Cinta – Numata
Rasa Ini – Vierra
Rindu 1/2 Mati – D’MASIV
Risalah Hati – Dewa
Roman Picisan – Dewa 19
Saat Bahagia – Ungu
Sahabat – Noah
Sakit Hati – Tipe-X
"""

daftar_target = []
baris_bersih = [b.strip() for b in DATA_LAGU_RAW.strip().split('\n') if b.strip()]

for i, baris in enumerate(baris_bersih):
    # Memproses format "Judul - Artis"
    if ' – ' in baris: sep = ' – '
    elif ' - ' in baris: sep = ' - '
    else: sep = None

    if sep:
        judul, artis = baris.split(sep, 1)
    else:
        judul, artis = baris, "Unknown"

    daftar_target.append({
        'Rank': i + 1,
        'Artis': artis.strip(),
        'Judul': judul.strip(),
        'Target': f"{artis.strip()} - {judul.strip()} (Official Audio)"
    })

log("SETUP", f"Berhasil memuat {len(daftar_target)} lagu custom.")

# ==========================================
# 2. SMART CACHE DOWNLOADER
# ==========================================
log("DOWNLOAD", "Sinkronisasi ke Master Database Drive...")
for item in daftar_target:
    safe_name = f"{item['Artis']} - {item['Judul']}".replace('/', '_').replace(':', '')
    path_master = os.path.join(DIR_DRIVE_MASTER, f"{safe_name}.mp3")
    path_lokal = os.path.join(DIR_DL, f"{item['Rank']:03d}_{safe_name}.mp3")

    if os.path.exists(path_master):
        shutil.copy(path_master, path_lokal)
    else:
        log("DOWNLOAD", f"⬇️ Mendownload: {safe_name}")
        subprocess.run(['yt-dlp', '-x', '--audio-format', 'mp3', '-o', path_lokal, f"ytsearch1:{item['Target']}"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if os.path.exists(path_lokal): shutil.copy(path_lokal, path_master)

# ==========================================
# 3. PABRIKAN VIDEO MASSAL
# ==========================================
log("FACTORY", "Memulai perakitan video...")
files_mp3 = [f for f in os.listdir(DIR_DL) if f.endswith('.mp3')]
footages = [f for f in os.listdir(DIR_FOOTAGE) if f.lower().endswith(('.mp4', '.mov', '.mkv'))]

for v in range(1, JUMLAH_VERSI + 1):
    log("BATCH", f"========== MERAKIT VERSI {v} ==========")
    video_input = os.path.join(DIR_FOOTAGE, random.choice(footages)) if footages else None
    playlist = random.sample(files_mp3, min(TOTAL_LAGU_PER_VIDEO, len(files_mp3)))

    path_audio_tmp = os.path.join(DIR_DL, f"temp_v{v}.mp3")
    path_txt = os.path.join(DIR_DL, f"list_v{v}.txt")
    teks_timeline = "00:00 - Start\n"
    total_detik = 0

    with open(path_txt, 'w', encoding='utf-8') as f:
        for lagu in playlist:
            f.write(f"file '{lagu.replace("'", "'\\''")}'\n")
            judul_bersih = lagu.split('_', 1)[-1].replace('.mp3','')
            teks_timeline += f"{format_waktu(total_detik)} - {judul_bersih}\n"
            try: total_detik += MP3(os.path.join(DIR_DL, lagu)).info.length
            except: pass

    # Concat Instan
    subprocess.run(['ffmpeg', '-f', 'concat', '-safe', '0', '-i', path_txt, '-c', 'copy', path_audio_tmp, '-y'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    base_name = f"VIDEO_NOSTALGIA_VOL_{v}_{WAKTU_PRODUKSI}"
    final_video_name = f"{base_name}.mp4"
    path_video_lokal = os.path.join(DIR_DL, final_video_name)

    if video_input:
        log("FFMPEG", "Muxing Video (Flash Speed Mode)...")
        subprocess.run([
            'ffmpeg', '-stream_loop', '-1', '-i', video_input, '-i', path_audio_tmp,
            '-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', '-shortest', '-fflags', '+genpts', path_video_lokal, '-y'
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Simpan ke Drive
    if os.path.exists(path_video_lokal):
        shutil.move(path_video_lokal, os.path.join(DIR_DRIVE_PROD, final_video_name))
        meta_json = {"title": base_name, "description": teks_timeline, "tags": ["nostalgia", "indonesia", "2000an"]}
        with open(os.path.join(DIR_DRIVE_PROD, f"{base_name}.json"), 'w', encoding='utf-8') as fj:
            json.dump(meta_json, fj, indent=4)
        log("DRIVE", f"✅ {final_video_name} Siap!")

log("DONE", f"🎉 Selesai! Cek folder hasil_produksi/{NAMA_FOLDER} di Drive Anda.")

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
[05:26:25] [SETUP] Menyiapkan Batch Produksi: PRODUKSI_NOSTALGIA_20260312_0526...
[05:26:25] [SETUP] Berhasil memuat 250 lagu custom.
[05:26:25] [DOWNLOAD] Sinkronisasi ke Master Database Drive...
[05:26:25] [DOWNLOAD] ⬇️ Mendownload: Gigi - 11 Januari
[05:26:37] [DOWNLOAD] ⬇️ Mendownload: Radja - Aku Ada Karena Kau Ada
[05:26:51] [DOWNLOAD] ⬇️ Mendownload: NaFF - Akhirnya Ku Menemukanmu
[05:27:05] [DOWNLOAD] ⬇️ Mendownload: Gigi - Andai
[05:27:20] [DOWNLOAD] ⬇️ Mendownload: Ari Lasso - Arti Cinta
[05:27:32] [DOWNLOAD] ⬇️ Mendownload: POTRET - Bagaikan Langit
[05:27:41] [DOWNLOAD] ⬇️ Mendownload: Acha Septriasa - Berdua Lebih Baik
[05:27:52] [DOWNLOAD] ⬇️ Mendownload: Sheila On 7 - Betapa
[05:28:05] [DOWNLOAD] ⬇️ Mendownload: Nidji - Bila Aku Jatuh Cinta
[05:28:15] [DOWNLOAD] ⬇️ 

# Bagian Baru

In [ ]:
# ==========================================
# 1. INITIAL SETUP & SYSTEM CORE
# ==========================================
import os, sqlite3, json, time, datetime, shutil, re, subprocess, random
import threading
from threading import Thread
from PIL import Image, ImageDraw, ImageFont

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    !pip install -q streamlit pyngrok google-api-python-client google-auth-oauthlib google-auth-httplib2 mutagen pillow groq
except: pass

from pyngrok import ngrok
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

# --- JALUR FOLDER DRIVE ---
NEXUS_BASE = "/content/drive/MyDrive/Nexus_Central"
DB_PATH = os.path.join(NEXUS_BASE, "nexus_vault.db")
DIR_CRED = os.path.join(NEXUS_BASE, "credentials")
DIR_THUMB = os.path.join(NEXUS_BASE, "thumbnails")
DIR_SUCCESS = os.path.join(NEXUS_BASE, "SUCCESS")
DIR_PROD_ROOT = "/content/drive/MyDrive/yt_mp3/hasil_produksi"
DIR_PROJ = os.path.join(NEXUS_BASE, "projects")
DIR_FONT = "/content/drive/MyDrive/yt_mp3/font" # Folder Font & Background

for d in [NEXUS_BASE, DIR_CRED, DIR_THUMB, DIR_SUCCESS, DIR_PROD_ROOT, DIR_PROJ, DIR_FONT]:
    os.makedirs(d, exist_ok=True)

def init_db():
    conn = sqlite3.connect(DB_PATH, timeout=30)
    conn.execute('''CREATE TABLE IF NOT EXISTS queue (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    project_name TEXT, file_name TEXT, file_path TEXT,
                    title TEXT, description TEXT, tags TEXT,
                    thumbnail_path TEXT, target_account TEXT,
                    status TEXT, publish_time TEXT, category_id TEXT)''')
    conn.commit(); conn.close()

init_db()

# ==========================================
# 2. GENERATOR DASHBOARD (app.py)
# ==========================================
with open('app.py', 'w', encoding='utf-8') as f:
    f.write("""
import streamlit as st
import sqlite3, os, datetime, json, re, random
from groq import Groq
from PIL import Image, ImageDraw, ImageFont
from google_auth_oauthlib.flow import InstalledAppFlow

os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'
st.set_page_config(page_title="Nexus Commander V11.12", layout="wide")

DIR_FONT = "/content/drive/MyDrive/yt_mp3/font"
DIR_THUMB = \"""" + DIR_THUMB + """\"

def run_query(query, params=(), commit=False):
    conn = sqlite3.connect(\"""" + DB_PATH + """\", timeout=60)
    cursor = conn.cursor()
    cursor.execute(query, params)
    res = cursor.fetchall()
    if commit: conn.commit()
    conn.close()
    return res

# --- SISTEM GENERATE THUMBNAIL OTOMATIS (MIRROR & POLOS) ---
def auto_generate_thumbnail(description_text, output_path):
    # 1. Ekstrak Lagu dari Timeline
    lines = description_text.strip().split('\\n')
    tracklist = []

    for line in lines:
        if " - " in line:
            parts = line.split(" - ", 1)
            song_name = parts[1].strip()
            if song_name.lower() != "start" and song_name != "":
                tracklist.append(song_name)

    tracks_to_show = tracklist[:26] # Batasi max 26 agar muat

    # 2. Siapkan Font
    font_list_path = os.path.join(DIR_FONT, "BebasNeue-Regular.ttf")
    try: f_list = ImageFont.truetype(font_list_path, 21)
    except: f_list = ImageFont.load_default()

    # 3. Siapkan Background Random (Tiap ditekan akan ngacak lagi)
    try:
        image_files = [f for f in os.listdir(DIR_FONT) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if image_files:
            random_img = random.choice(image_files)
            bg_path = os.path.join(DIR_FONT, random_img)
            img = Image.open(bg_path).convert("RGBA").resize((1280, 720))
        else:
            img = Image.new('RGBA', (1280, 720), (50, 40, 30, 255))
    except:
        img = Image.new('RGBA', (1280, 720), (50, 40, 30, 255))

    txt_layer = Image.new('RGBA', img.size, (255, 255, 255, 0))
    d = ImageDraw.Draw(txt_layer)

    # 4. Gambar Teks Sisi Kiri
    list_y = 40
    list_start_x_left = 30
    for i, track in enumerate(tracks_to_show):
        idx = i + 1
        if len(track) > 55: track = track[:52] + "..."
        txt = f"{idx:02d}. {track}"
        d.text((list_start_x_left, list_y), txt, font=f_list, fill=(255, 255, 255, 255))
        list_y += 25

    # 5. Gambar Teks Sisi Kanan (Mirrored Layout)
    list_y = 40
    for i, track in enumerate(tracks_to_show):
        idx = i + 1
        if len(track) > 55: track = track[:52] + "..."
        txt = f"{track} .{idx:02d}"

        try:
            bbox_txt = d.textbbox((0,0), txt, font=f_list)
            txt_w = bbox_txt[2] - bbox_txt[0]
        except: txt_w = len(txt) * 9

        list_start_x_right = 1250 - txt_w
        if list_start_x_right < 650: list_start_x_right = 650

        d.text((list_start_x_right, list_y), txt, font=f_list, fill=(255, 255, 255, 255))
        list_y += 25

    # 6. Simpan Final
    final_img = Image.alpha_composite(img, txt_layer).convert("RGB")
    final_img.save(output_path, quality=95)
    return True

# --- AI BRAIN ---
def expand_seo_groq(api_key, current_title, current_text, v, identitas, custom_prompt):
    client = Groq(api_key=api_key)
    prompt = custom_prompt.replace("{current_title}", current_title).replace("{current_text}", current_text).replace("{v}", str(v)).replace("{identitas}", identitas)
    chat = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(chat.choices[0].message.content)

def smart_pairing(video_path):
    base_dir = os.path.dirname(video_path)
    fname_no_ext = os.path.splitext(os.path.basename(video_path))[0]
    m_json = os.path.join(base_dir, f"{fname_no_ext}.json")
    auto_thumb = os.path.join(\"""" + DIR_THUMB + """\", f"{fname_no_ext}.jpg")
    title, desc, tags = "", "", ""
    found_json = False

    if os.path.exists(m_json):
        try:
            with open(m_json, 'r', encoding='utf-8') as fj:
                d = json.load(fj)
                title, desc = d.get('title', f"{fname_no_ext}"), d.get('description', '')
                rt = d.get('tags', 'musik')
                tags = ", ".join(rt) if isinstance(rt, list) else str(rt)
                found_json = True
        except: pass
    else:
        title = fname_no_ext
        desc = "00:00 - Start\\n00:00 - Kosong - Belum ada Timeline"
        tags = "musik"

    thumb = auto_thumb if os.path.exists(auto_thumb) else ""
    return title, desc, tags, thumb, "READY" if found_json else "DRAFT"

st.title("🚀 Nexus Command Center V11.12")

with st.sidebar:
    st.header("📂 Project Manager")
    new_p = st.text_input("Buat Proyek Baru:")
    if st.button("Tambah Proyek"):
        if new_p:
            os.makedirs(os.path.join(\"""" + DIR_PROJ + """\", new_p), exist_ok=True)
            st.success(f"Proyek {new_p} Aktif!"); st.rerun()

    st.divider()
    groq_key = st.text_input("Groq API Key", type="password")
    projs = sorted([d for d in os.listdir(\"""" + DIR_PROJ + """\") if os.path.isdir(os.path.join(\"""" + DIR_PROJ + """\", d))])
    sel_proj = st.selectbox("Pilih Proyek", ["---"] + projs)

    st.divider()
    secrets = sorted([f for f in os.listdir(\"""" + DIR_CRED + """\") if f.startswith("secret_")])
    sel_sec = st.selectbox("API Secret (Bensin)", secrets if secrets else ["Missing Secret"])

    if st.button("🗑️ Reset Database"):
        run_query("DELETE FROM queue", commit=True); st.rerun()

t1, t2, t3, t4, t5 = st.tabs(["🔍 SCAN", "🧠 AI LAB", "👤 ACC", "🚀 LAUNCH", "📜 LOGS"])

with t1:
    mode = st.radio("Mode Pencarian Folder", ["Otomatis (Produksi Terbaru)", "Manual (Custom Path)"], horizontal=True)
    if mode == "Otomatis (Produksi Terbaru)":
        subs = [os.path.join(\"""" + DIR_PROD_ROOT + """\", d) for d in os.listdir(\"""" + DIR_PROD_ROOT + """\") if os.path.isdir(os.path.join(\"""" + DIR_PROD_ROOT + """\", d))]
        scan_path = max(subs, key=os.path.getmtime) if subs else ""
        st.info(f"Mendeteksi: {os.path.basename(scan_path) if scan_path else 'Tidak ada'}")
    else:
        scan_path = st.text_input("Jalur Folder Manual:", value="/content/drive/MyDrive/yt_mp3/hasil")

    if st.button("🔎 Scan Video"):
        if os.path.exists(scan_path):
            files = [f for f in os.listdir(scan_path) if f.endswith(".mp4")]
            for f in files:
                if not run_query("SELECT id FROM queue WHERE file_name=?", (f,)):
                    v_p = os.path.join(scan_path, f)
                    l_t, l_d, l_tg, l_th, stat = smart_pairing(v_p)
                    run_query("INSERT INTO queue (project_name, file_name, file_path, title, description, tags, thumbnail_path, status) VALUES (?,?,?,?,?,?,?,?)",
                             (sel_proj, f, v_p, l_t, l_d, l_tg, l_th, stat), commit=True)
            st.success("Sinkronisasi Selesai!")
        else: st.error("Folder tidak ditemukan!")

with t2:
    with st.expander("⚙️ Edit AI Prompt Template", expanded=False):
        default_p = '''TUGAS: Pakar SEO YouTube Musik Kelas Dunia.
BAHAN JUDUL (Ketikan User): "{current_title}"
BAHAN DESKRIPSI & TIMELINE: "{current_text}"
INFO: VOL {v} | {identitas}

ATURAN MATI:
bahasa ingris
1. JUDUL:
   - Jika 'BAHAN JUDUL' berisi nama mentah (misal: VIDEO_FINAL...),
   - PENTING: JUDUL MAKSIMAL 90 KARAKTER!
buat judul clickbait: contoh [EMOJI] TOP SPOTIFY 2026 VOL {v} | [Ambil 5 Nama Artis ASLI dari Timeline].
Top World Hits 2026 🌍 Lagu Internasional Paling Enak Didengar
Best World Songs Playlist 🎶 Lagu Barat Terbaik Sepanjang Waktu
Global Hits Playlist 🌎 Lagu Viral Dunia yang Wajib Didengar
Top International Music 2026 🔥 Lagu Barat Paling Populer
World Music Vibes ✨ Lagu Internasional Terbaik Nonstop
Chill World Songs 🌙 Lagu Barat Enak Didengar Saat Santai
Late Night World Playlist 🌌 Lagu Internasional Buat Temani Malam

Relaxing International Songs ☕ Playlist Lagu Barat Cozy

Judul untuk Lagu Viral / Trending

Viral World Songs 2026 🚀 Lagu Barat yang Lagi Trending

TikTok & Global Hits 🎵 Lagu Viral Dunia Terbaru

Judul yang SEO Friendly (bagus untuk YouTube)

Top 50 World Songs 2026 🌍 Best English Songs Playlist

Best English Songs Playlist 2026 🎧 Nonstop Popular Hits

Top Global Hits 2026 🔥 Lagu Barat Terpopuler Saat Ini
2. DESKRIPSI: Buat 150 kata paragraf SEO High-Keyword yang nyambung dengan judul. LALU WAJIB SALIN 100% TIMELINE dari bahan (TANPA UBAH ANGKA SEDIKITPUN).
perhatikan enter, buat se engage mugkin
3. TAGS: 25 keyword trending.
OUTPUT WAJIB JSON MURNI: { "title":"", "description":"", "tags":[] }'''

        st.session_state.custom_prompt = st.text_area("🔧 Groq SEO Prompt", value=st.session_state.get('custom_prompt', default_p), height=350)

    drafts = run_query("SELECT file_name FROM queue WHERE status != 'SUCCESS'")
    if drafts:
        sel_f = st.selectbox("Pilih Video", [d[0] for d in drafts])
        data = run_query("SELECT * FROM queue WHERE file_name=?", (sel_f,))[0]

        c1, c2 = st.columns(2)
        with c1:
            u_t = st.text_input("Judul (Edit atau biarkan AI yang buat)", value=data[4])
            u_d = st.text_area("Deskripsi/Timeline (Data Asli)", value=data[5], height=350)
            u_tg = st.text_input("Tags", value=data[6])
        with c2:
            st.write("🖼️ **Thumbnail**")

            # Tampilkan Thumbnail Jika Ada
            # Ditambahkan opsi timestamp agar preview streamlit selalu refresh ketika di generate ulang
            if data[7] and os.path.exists(data[7]):
                try:
                    # Membaca sebagai bytes agar streamtlit tidak menggunakan cache gambar lama
                    from PIL import Image
                    img_thumb = Image.open(data[7])
                    st.image(img_thumb, width=300)
                except: st.error("Thumbnail Rusak!")
            else:
                st.info("Belum ada thumbnail.")

            # FITUR BARU: REGENERATE AUTO-THUMBNAIL
            st.markdown("---")
            st.info("💡 **Tips:** Tekan terus tombol di bawah ini sampai Anda mendapatkan Background yang pas!")
            if st.button("🔄 Auto-Generate / Regenerate Thumbnail"):
                t_p = os.path.join(\"""" + DIR_THUMB + """\", f"{sel_f}.jpg")
                if auto_generate_thumbnail(u_d, t_p):
                    run_query("UPDATE queue SET thumbnail_path=? WHERE file_name=?", (t_p, sel_f), commit=True)
                    st.success("Thumbnail diperbarui dengan komposisi baru!"); st.rerun()

            st.write("Atau masukkan Custom Cover Image:")
            u_thumb = st.file_uploader("Upload Manual (JPG)", type=['jpg', 'jpeg'])
            if u_thumb:
                t_p = os.path.join(\"""" + DIR_THUMB + """\", f"{sel_f}.jpg")
                img = Image.open(u_thumb).convert("RGB")
                img.save(t_p, "JPEG", optimize=True, quality=85)
                run_query("UPDATE queue SET thumbnail_path=? WHERE file_name=?", (t_p, sel_f), commit=True); st.rerun()

            st.write("---")
            if st.button("🪄 Expand AI SEO (Baca Judul & Timeline)"):
                if groq_key:
                    with st.spinner("AI Sedang Meracik..."):
                        vol = re.search(r'VOL[_\\s-]*([0-9]+)', sel_f, re.IGNORECASE).group(1) if "VOL" in sel_f.upper() else "1"
                        ident = "INDONESIA" if "ID" in sel_f.upper() else "GLOBAL"
                        res = expand_seo_groq(groq_key, u_t, u_d, vol, ident, st.session_state.custom_prompt)
                        run_query("UPDATE queue SET title=?, description=?, tags=?, status='READY' WHERE file_name=?",
                                 (res['title'], res['description'], ", ".join(res['tags']), sel_f), commit=True); st.rerun()

        if st.button("💾 SIMPAN MANUAL KE READY"):
            run_query("UPDATE queue SET title=?, description=?, tags=?, status='READY' WHERE file_name=?", (u_t, u_d, u_tg, sel_f), commit=True); st.rerun()
    else: st.info("Antrean kosong.")

with t3:
    n_acc = st.text_input("Nama Akun")
    if st.button("🔑 Link Login YouTube"):
        flow = InstalledAppFlow.from_client_secrets_file(os.path.join(\"""" + DIR_CRED + """\", sel_sec),
               scopes=["https://www.googleapis.com/auth/youtube.upload"], redirect_uri='http://localhost')
        st.session_state.flow = flow
        st.code(flow.authorization_url(prompt='consent')[0])
    l_u = st.text_input("Paste URL Localhost:")
    if st.button("✅ Save Token Akun"):
        if 'flow' in st.session_state:
            st.session_state.flow.fetch_token(authorization_response=l_u)
            with open(os.path.join(\"""" + DIR_CRED + """\", f"token_{n_acc}.json"), 'w') as f: f.write(st.session_state.flow.credentials.to_json())
            st.success("Berhasil!"); del st.session_state.flow

with t4:
    ready = run_query("SELECT id, file_name FROM queue WHERE status='READY'")
    if ready:
        with st.form("launch_form"):
            selected = [r[0] for r in ready if st.checkbox(f"{r[1]}", key=f"sel_{r[0]}")]
            accs = [f.replace("token_","").replace(".json","") for f in os.listdir(\"""" + DIR_CRED + """\") if f.startswith("token_")]
            t_acc = st.selectbox("Pilih Channel", accs)
            gap = st.number_input("Interval (Jam)", 1, 24, 12)
            s_d = st.date_input("Mulai Tanggal")
            s_t = st.time_input("Mulai Jam")
            if st.form_submit_button("🚀 START UPLOAD SELECTION"):
                comb = datetime.datetime.combine(s_d, s_t)
                for i, r_id in enumerate(selected):
                    p_t = (comb + datetime.timedelta(hours=i*gap)).isoformat() + "Z"
                    run_query("UPDATE queue SET status='QUEUED', target_account=?, publish_time=? WHERE id=?", (t_acc, p_t, r_id), commit=True)
                st.success("Antrean diamankan! Pantau Terminal Colab."); st.rerun()

with t5:
    col_r, _ = st.columns([1,4])
    with col_r:
        if st.button("🔄 Segarkan Data Log"):
            st.rerun()

    logs = run_query("SELECT file_name, status, target_account, publish_time FROM queue ORDER BY id DESC LIMIT 50")
    if logs:
        for l in logs:
            st.markdown(f"**{l[0]}**")
            if "UPLOADING" in str(l[1]) and "(" in str(l[1]) and "%" in str(l[1]):
                m = re.search(r'\\((\\d+)%\\)', l[1])
                if m:
                    pct = int(m.group(1))
                    if pct > 100: pct = 100
                    st.progress(pct, text=f"Sedang Mengunggah... {pct}%")
                else: st.markdown(f"{l[1]}")
            else: st.markdown(f"{l[1]}")
            st.markdown(f"Akun {l[2]} katakata")
            st.markdown(f"{l[3]}")
            st.divider()
    else: st.info("Log Kosong.")
    """)

# ==========================================
# 3. IRONCLAD UPLOADER ENGINE (ANTI-DRIVE DISCONNECT)
# ==========================================
def run_uploader_engine():
    print("\\n" + "="*50)
    print("🛰️ NEXUS DISPATCHER V11.12 - PERFECT BRAIN EDITION")
    print("="*50 + "\\n")

    while True:
        conn = None
        try:
            conn = sqlite3.connect(DB_PATH, timeout=60)
            conn.execute("BEGIN IMMEDIATE")
            job = conn.execute("SELECT * FROM queue WHERE status='QUEUED' ORDER BY id ASC LIMIT 1").fetchone()

            if job:
                jid, proj, fname, fpath, title, desc, tags, th_path, acc, _, pt, _ = job
                conn.execute("UPDATE queue SET status='UPLOADING (0%)' WHERE id=?", (jid,))
                conn.commit()

                now = datetime.datetime.now().strftime("%H:%M:%S")
                print(f"[{now}] 🔒 MENGUNCI & MENGUNGGAH: {fname}")

                safe_title = title if title and str(title).strip() != "" else fname
                safe_title = safe_title[:95]
                safe_desc = desc if desc else "Upload by Nexus Commander"
                safe_tags = tags.split(',') if tags else ["music", "nexus"]

                secrets = sorted([f for f in os.listdir(DIR_CRED) if f.startswith("secret_")])
                success = False

                # SOLUSI ERROR 107 (TRANSPORT ENDPOINT): COPY DULU KE LOKAL MEMORI COLAB
                temp_local_path = f"/content/temp_{jid}.mp4"
                print(f"[{now}] 📥 Menyalin video ke memori lokal Colab agar koneksi stabil...")
                try:
                    shutil.copy2(fpath, temp_local_path)
                except Exception as e:
                    print(f"[{now}] ❌ Gagal salin file: {e}. Coba hubungkan ulang Drive!")
                    conn.execute("UPDATE queue SET status='READY' WHERE id=?", (jid,))
                    conn.commit(); conn.close()
                    continue

                for sec in secrets:
                    try:
                        t_file = os.path.join(DIR_CRED, f"token_{acc}.json")
                        creds = Credentials.from_authorized_user_file(t_file, ["https://www.googleapis.com/auth/youtube.upload"])
                        if creds.expired: creds.refresh(Request())
                        yt = build("youtube", "v3", credentials=creds)

                        body = {
                            'snippet': {'title': safe_title, 'description': safe_desc, 'tags': safe_tags, 'categoryId': '10'},
                            'status': {'privacyStatus': 'private', 'publishAt': pt, 'selfDeclaredMadeForKids': False}
                        }

                        print(f"[{now}] ⬆️ API Proses Data: {fname}")

                        chunk_size = int(5 * 1024 * 1024)
                        media = MediaFileUpload(temp_local_path, chunksize=chunk_size, resumable=True) # Pakai file lokal!

                        request_insert = yt.videos().insert(part="snippet,status", body=body, media_body=media)
                        response = None

                        while response is None:
                            upload_status, response = request_insert.next_chunk()
                            if upload_status:
                                pct = int(upload_status.progress() * 100)
                                print(f"[{now}] ⏳ Mengunggah... {pct}%")
                                try:
                                    conn_up = sqlite3.connect(DB_PATH, timeout=60)
                                    conn_up.execute("UPDATE queue SET status=? WHERE id=?", (f"UPLOADING ({pct}%)", jid))
                                    conn_up.commit(); conn_up.close()
                                except: pass

                        resp = response

                        if th_path and os.path.exists(th_path):
                            print(f"[{now}] 🖼️ Mengunggah Thumbnail...")
                            try: yt.thumbnails().set(videoId=resp['id'], media_body=MediaFileUpload(th_path)).execute()
                            except Exception as th_err: print(f"[{now}] ⚠️ Thumbnail gagal: {th_err}")

                        success = True
                        print(f"[{now}] ✅ SUKSES! Video terkirim.")
                        break

                    except Exception as e:
                        if "quotaExceeded" in str(e):
                            print(f"[{now}] ⚠️ Quota limit! Pindah secret...")
                            continue
                        else:
                            print(f"[{now}] ❌ ERROR: {e}")
                            break

                # PEMBERSIHAN MEMORI SEMENTARA
                if os.path.exists(temp_local_path):
                    os.remove(temp_local_path)
                    print(f"[{now}] 🧹 Membersihkan cache lokal.")

                if success:
                    conn.execute("UPDATE queue SET status='SUCCESS' WHERE id=?", (jid,))
                    if os.path.exists(fpath): shutil.move(fpath, os.path.join(DIR_SUCCESS, fname))
                else:
                    conn.execute("UPDATE queue SET status='READY' WHERE id=?", (jid,))
                conn.commit()
            else: conn.commit()

            conn.close()
        except Exception as e:
            print(f"⚠️ SISTEM ERROR DATABASE: {e}")
            if conn: conn.close()

        time.sleep(10)

# ==========================================
# 4. RUNNER (ANTI-DUPLICATE & ANTI-RELOAD)
# ==========================================
NGROK_TOKEN = "3AhwkLi59oy3rvXauHoe35i31Gw_3T5EoHrj9wGSUvvvqYCZb"
ngrok.set_auth_token(NGROK_TOKEN)

is_running = any(t.name == "NexusWorker" for t in threading.enumerate())
if not is_running:
    t = Thread(target=run_uploader_engine, name="NexusWorker", daemon=True)
    t.start()
    print("✅ Supir Uploader Utama diaktifkan.")
else: print("ℹ️ Supir Uploader sudah berjalan.")

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.runOnSave", "false", "--server.fileWatcherType", "none"])

url = ngrok.connect(8501).public_url
print(f"\\n📱 DASHBOARD NEXUS V11.12 LIVE:\\n {url} \\n")
while True: time.sleep(100)


⚠️ SISTEM ERROR DATABASE: unable to open database file
Mounted at /content/drive
[07:35:15] 🔒 MENGUNCI & MENGUNGGAH: VIDEO_NOSTALGIA_VOL_3_20260312_0526.mp4
[07:35:15] 📥 Menyalin video ke memori lokal Colab agar koneksi stabil...
ℹ️ Supir Uploader sudah berjalan.
\n📱 DASHBOARD NEXUS V11.12 LIVE:\n https://trinidad-crustaceous-nonnocturnally.ngrok-free.dev \n
[07:35:15] ⬆️ API Proses Data: VIDEO_NOSTALGIA_VOL_3_20260312_0526.mp4
[07:35:15] ❌ ERROR: <HttpError 400 when requesting None returned "The request metadata specifies invalid video keywords.". Details: "[{'message': 'The request metadata specifies invalid video keywords.', 'domain': 'youtube.video', 'reason': 'invalidTags', 'location': 'body.snippet.tags', 'locationType': 'other'}]">
[07:35:15] 🧹 Membersihkan cache lokal.
[07:36:32] 🔒 MENGUNCI & MENGUNGGAH: VIDEO_NOSTALGIA_VOL_4_20260312_0526.mp4
[07:36:32] 📥 Menyalin video ke memori lokal Colab agar koneksi stabil...
[07:36:32] ⬆️ API Proses Data: VIDEO_NOSTALGIA_VOL_4_20260312_0

KeyboardInterrupt: 